In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    # M × I (Market × Interest Rate)
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    # M × P (Market × Price)
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    # M × V (Market × Volatility)
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    # Sharpe-like ratio
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    # M × S (Market × Sentiment)
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    # E × I (Economic × Interest Rate)
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    # E × P (Economic × Price)
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    # E × V (Economic × Volatility)
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    # E × S (Economic × Sentiment)
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    # I × P (Interest Rate × Price)
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    # I × V (Interest Rate × Volatility)
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    # I × S (Interest Rate × Sentiment)
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    # P × V (Price × Volatility)
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    # P × S (Price × Sentiment)
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    # Contrarian signal (opposite signs amplify)
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    # V × S (Volatility × Sentiment)
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    # M × E × I
    data[f'MEI_M*_E*_I*'] = (data['M*'] * data['E*']) / (data['I*'] + 1e-8)
    main_features.append(f'MEI_M*_E*_I*')
    # M × P × V
    data[f'MPV_M*_P*_V*'] = (data['M*'] / (data['V*'] + 1e-8)) * (1 / (data['P*'] + 1e-8))
    main_features.append(f'MPV_M*_P*_V*')
    # E × I × P
    data[f'EIP_E*_I*_P*'] = (data['E*'] - data['I*']) * data['P*']
    main_features.append(f'EIP_E*_I*_P*')
    # M × S × V
    data[f'MSV_M*_S*_V*'] = (data['M*'] * data['S*']) / (data['V*'] + 1e-8)
    main_features.append(f'MSV_M*_S*_V*')
    # E × P × S
    data[f'EPS_gap_E*_P*_S*'] = (data['E*'] * data['P*']) - data['S*']
    main_features.append(f'EPS_gap_E*_P*_S*')

    # D × M
    data[f'regime_D*_M*'] = data['D*'] * data['M*']
    main_features.append(f'regime_D*_M*')
    # D × E
    data[f'regime_D*_E*'] = data['D*'] * data['E*']
    main_features.append(f'regime_D*_E*')
    # D × V
    data[f'regime_D*_V*'] = data['D*'] * data['V*']
    main_features.append(f'regime_D*_V*')
    # D × S
    data[f'regime_D*_S*'] = data['D*'] * data['S*']
    main_features.append(f'regime_D*_S*')
    # Complex regime: D × D × Feature
    regime = data['D*'] * data['D*']
    data[f'complex_regime_D*_D*_M*'] = regime * data['M*']
    main_features.append(f'complex_regime_D*_D*_M*')
    # Z-scores
    z_scores = {}
    for category in ['M', 'E', 'I', 'P', 'V', 'S']:
        col = f'{category}*'
        z_scores[col] = (data[col] - data[col].mean()) / (data[col].std() + 1e-8)
    # M × S z-score product
    data[f'z_prod_M*_S*'] = z_scores['M*'] * z_scores['S*']
    main_features.append(f'z_prod_M*_S*')
    # P × V z-score product
    data[f'z_prod_P*_V*'] = z_scores['P*'] * z_scores['V*']
    main_features.append(f'z_prod_P*_V*')
    
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,MSV_M*_S*_V*,EPS_gap_E*_P*_S*,regime_D*_M*,regime_D*_E*,regime_D*_V*,regime_D*_S*,complex_regime_D*_D*_M*,z_prod_M*_S*,z_prod_P*_V*,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,-0.140065,26.917739,0.398262,-7.737146,-4.958923,-1.744000,-0.398262,0.254204,0.052235,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.095828,47.110238,0.205104,7.796466,4.714755,2.202814,0.205104,0.116190,0.513803,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,-0.555413,53.641566,-1.891902,15.645830,9.723650,5.709218,-3.783803,0.031223,0.738451,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,-1.982602,51.513256,-7.862249,27.061086,13.188988,9.977494,-23.586747,-0.144422,0.372932,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,-1.206518,46.876079,-3.572073,17.586497,8.748778,5.910046,-7.144147,0.007032,0.308746,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.774132,36.591798,0.000000,0.000000,0.000000,0.000000,0.000000,0.008344,0.060698,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,3.385677,34.124670,0.000000,0.000000,0.000000,0.000000,0.000000,0.068458,0.019910,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,2.801633,23.913798,1.803055,6.994756,2.764110,4.294945,1.803055,-0.002385,-0.027406,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,0.606036,43.926840,0.000000,0.000000,0.000000,0.000000,0.000000,0.091636,-0.082909,0.008310


In [4]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBM_R_model = joblib.load('LGBM_R.pkl')

LGBMRegressor TRAINING...


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.45328218141009147


In [6]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))